# MLP (sklearn) (multikelas)
**tabular › multiclass** · notebook `mlp`

**Kapan dipakai:** Pembanding jaringan saraf sederhana, bagus untuk blend dengan model pohon. Di sini 7 kelas dengan distribusi tidak seimbang (covertype): perhatikan F1 macro, bukan hanya accuracy.

**Jebakan umum:** Wajib scaling. Hasil bervariasi antar seed; di tabular biasanya kalah dari boosting.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular_extra/multiclass/covertype_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular_extra/multiclass/covertype_test.csv"      # None kalau tidak ada file test
TARGET     = "Cover_Type"
ID_COL     = 'id'                  # kolom ID untuk submission (None kalau tidak ada)
DROP_COLS  = []                  # bocoran target, ID tambahan, dll
GROUP_COL  = None                            # isi bila ada grup (pasien/pelanggan sama) agar tidak bocor
SCORING    = "f1_macro"                # metrik CV (sklearn scoring name)
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "mlp"
TASK       = "classification"
OUT = out_dir("tabular", "multiclass", MODEL_KEY)
set_seed(SEED)

## 2. Baca data & lihat sekilas

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head())
display(df.isna().mean().round(3).to_frame("missing").T)
display(df[TARGET].describe() if TASK == "regression" else df[TARGET].value_counts(normalize=True).to_frame("proporsi"))

## 3. Pilih fitur
Kolom dipisah otomatis jadi numerik / kategorikal / tanggal. Periksa daftar buangan: kalau ada kolom bocoran target, masukkan ke `DROP_COLS`.

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score, feature_names
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df[TARGET])
print("kelas:", dict(enumerate(le.classes_)))
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
print(Xtr.shape, Xva.shape)

## 4. Model
Kode model di bawah ini bisa disalin langsung ke tempat lain.

In [ ]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(hidden_layer_sizes=(128, 64), early_stopping=True, max_iter=500, random_state=42)
pipe = build_pipeline(model, NUM, CAT, kind="nn")
pipe

## 5. Cross-validation (pada data training)
Ini perkiraan skor yang lebih jujur daripada satu kali split.

In [ ]:
folds, summary = cv_score(pipe, Xtr, ytr, TASK, SCORING, N_SPLITS, SEED, groups=None if groups is None else groups.iloc[Xtr.index])
display(folds.round(4)); print(summary)

## 6. Latih & evaluasi pada validation

In [ ]:
pipe.fit(Xtr, ytr)
pred = pipe.predict(Xva)
proba = pipe.predict_proba(Xva) if hasattr(pipe, "predict_proba") else None
metrics = classification_metrics(yva, pred, proba, classes=np.arange(len(le.classes_)))
display(show_table(metrics))
from sklearn.metrics import classification_report
print(classification_report(yva, pred, target_names=[str(c) for c in le.classes_]))
plot_confusion(yva, pred, labels=np.arange(len(le.classes_)), path=OUT / "confusion.png"); plt.show()

## 7. Fitur mana yang penting?
Permutation importance: model-agnostik, dihitung pada validation. Cocok untuk bagian analisis di laporan.

In [ ]:
from sklearn.inspection import permutation_importance
pi = permutation_importance(pipe, Xva, yva, scoring=SCORING, n_repeats=5, random_state=SEED, n_jobs=1)
plot_importance(Xva.columns, pi.importances_mean, path=OUT / "importance.png", title="Permutation importance"); plt.show()

## 8. Latih ulang pada semua data → prediksi test → submission

In [ ]:
from sklearn.base import clone
final = clone(pipe).fit(X, y)
if test is not None:
    Xte, _, _ = prepare_features(test, cols)
    Xte = Xte.reindex(columns=X.columns)
    preds = le.inverse_transform(final.predict(Xte))
    ids = test[ID_COL] if ID_COL else np.arange(len(test))
    sub = make_submission(ids, preds, ID_COL or "id", TARGET, OUT / "submission.csv")
    display(sub.head())

## 9. Simpan & uji muat ulang

In [ ]:
save_model({"pipe": final, "cols": cols, "target": TARGET, "id_col": ID_COL, "classes": getattr(le, "classes_", None) if TASK == "classification" else None}, OUT / "model.joblib")
save_json({"cv": summary, "valid": metrics, "n_train": len(X), "model": MODEL_KEY}, OUT / "metrics.json")
bundle = load_model(OUT / "model.joblib")
again = bundle["pipe"].predict(Xva.reindex(columns=X.columns))
assert np.allclose(again, final.predict(Xva.reindex(columns=X.columns))), "model muat-ulang menghasilkan prediksi berbeda"
print("OK. hasil di", OUT)

## Langkah berikutnya
- Skor CV jauh di atas validation/leaderboard → curigai **kebocoran** (kolom turunan target, ID, waktu).
- Coba model lain di `notebooks/tabular/` (LightGBM, CatBoost) dan **blend** hasilnya (lihat `stacking`).
- Tuning: lihat `notebooks/tabular/tuning/`.
- Feature engineering & resep lain: `cookbook/`.